In [1]:
from scipy.optimize import minimize
import numpy as np
import autograd.numpy as anp
from autograd import grad, jacobian

In [2]:
pip install autograd

Note: you may need to restart the kernel to use updated packages.


### Problem Statement: The Factory Production Mix

A factory manufactures two products: Product A ($x_1$) and Product B ($x_2$).  

Because of overtime pay and machine wear-and-tear, the cost to produce these items scales non-linearly.   
Furthermore, producing them at the same time creates a slight logistical bottleneck, adding an interaction cost.  
  
The Goal: Determine exactly how many units of Product A and Product B to manufacture to minimize the total production cost, while meeting a specific client order and staying within the factory's machine-hour limits.    

The Parameters & Rules:  
Objective Function (Cost): Minimize the quadratic production and interaction cost given by $f(\theta) = 3x_1^2 + 2x_2^2 + x_1 x_2$.    

Contract Fulfillment: The client ordered exactly 20 units total ($x_1 + x_2 = 20$).  
Machine Time Limit: The factory has a maximum of 32 hours of machine time available.   
Product A takes 2 hours per unit, and Product B takes 1 hour per unit ($2x_1 + x_2 \le 32$).  
Production Bounds: The factory cannot produce negative units, and the packaging department can handle a maximum of 15 units of any single product type ($0 \le x_1 \le 15$ and $0 \le x_2 \le 15$).

#### Objective Function: minimize cost

$$\min_{\theta} f(\theta) = 3x_1^2 + 2x_2^2 + x_1 x_2$$

x1 = units of production of Product A  
x2 = units of production of Product B  
state vector theta = [x1,x2]

### Methods 1 to 3 Solve the Objective Function Using SciPy

### Method 1: Using SciPy and Not Passing in Gradient and Jacobian into Solver

In [3]:
def objective(theta):
    x1 = theta[0]
    x2 = theta[1]
    return 3*(x1**2) + 2*(x2**2) + x1*x2

#### Constraints

constraint 1: client ordered exactly 20 units total ==> equality constraint  
g1(theta): x1 + x2 - 20 = 0

In [4]:
def constraint1(theta):
    x1 = theta[0]
    x2 = theta[1]
    return x1 + x2 - 20

constraint 2: 32 hours max of machine time. Product A takes 2 hours per unit and product B takes 1 hour per unit ==> equality constraint  
g2(theta): 32 - 2x1 - x2 >= 0

In [5]:
def constraint2(theta):
    x1 = theta[0]
    x2 = theta[1]
    return 32 - 2*x1 - x2

In [6]:
con1 = {"type" : "eq", "fun" :constraint1}
con2 = {"type" : "ineq", "fun" : constraint2}
cons = [con1, con2]

#### Bounds

0 <= x1 <= 15  
0 <= x2 <= 15

In [7]:
bound1 = [0,15]
bound2 = [0,15]
bounds = [bound1,bound2]

#### Initial Guess

In [8]:
theta0 = [2,2]

In [9]:
sol = minimize(fun = objective,method = "SLSQP", x0 = theta0,bounds = bounds, constraints = cons)

In [10]:
print(sol)

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 575.0
           x: [ 7.500e+00  1.250e+01]
         nit: 4
         jac: [ 5.750e+01  5.750e+01]
        nfev: 14
        njev: 4
 multipliers: [ 5.750e+01  0.000e+00]


### Method 2: Using Autograd to Feed the Gradient and Jacobian

In [11]:
obj_grad = grad(objective)
con1_jac = jacobian(constraint1)
con2_jac = jacobian(constraint2)

In [12]:
constraints_with_jac = [
    {"type" : "eq", "fun" : constraint1, "jac" : con1_jac}, 
    {"type" : "ineq", "fun" : constraint2, "jac" : con2_jac}
]

In [13]:
sol2 = minimize(fun = objective, method = "SLSQP", x0 = theta0, bounds = bounds, constraints = constraints_with_jac)

In [14]:
print(sol2)

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 575.0
           x: [ 7.500e+00  1.250e+01]
         nit: 4
         jac: [ 5.750e+01  5.750e+01]
        nfev: 14
        njev: 4
 multipliers: [ 5.750e+01  0.000e+00]


### Method 3: Doing Calculus Myself

In [15]:
def objective_grad(theta):
    x1 = theta[0]
    x2 = theta[1]
    df_dx1 = 6 * x1 + x2
    df_dx2 = 4 * x2 + x1
    return np.array([df_dx1, df_dx2])
    

In [16]:
def constraint_1_jac(theta):
    x1 = theta[0]
    x2 = theta[1]
    df_dx1 = 1
    df_dx2 = 1
    return np.array([df_dx1, df_dx2])

In [17]:
def constraint_2_jac(theta):
    x1 = theta[0]
    x2 = theta[1]
    df_dx1 = -2
    df_dx2 = -1
    return np.array([df_dx1, df_dx2])

In [18]:
constraints_with_jac_2 = [
    {"type":"eq", "fun":constraint1, "jac":constraint_1_jac},
    {"type":"ineq", "fun":constraint2, "jac":constraint_2_jac}
]

In [19]:
sol3 = minimize(fun = objective, x0 = theta0, method = "SLSQP", bounds = bounds, jac = objective_grad, constraints = constraints_with_jac_2)

In [20]:
print(sol3)

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 574.9999999999999
           x: [ 7.500e+00  1.250e+01]
         nit: 4
         jac: [ 5.750e+01  5.750e+01]
        nfev: 6
        njev: 4
 multipliers: [ 5.750e+01  0.000e+00]


## Method 4:

In [21]:
from docplex.mp.model import Model

In [22]:
mdl = Model(name = "Factory")

In [23]:
x1 = mdl.integer_var(lb = 0, ub = 15, name = "Product_A")
x2 = mdl.integer_var(lb = 0, ub = 15, name = "Product_B")

mdl.add_constraint(x1 + x2 == 20, ctname = "Contract_Fullfilment")
mdl.add_constraint(2*x1 + x2 <= 32, ctname = "Machine_Hourse")

mdl.minimize(3*(x1**2) + 2*(x2**2) + x1*x2)
sol4 = mdl.solve()

print(sol4.get_value(x1))
print(sol4.get_value(x2))
print(sol4.get_objective_value())

8.0
12.0
576.0
